# Phase 7 — dual_attention_moe LoRMA prior case retrieval


In [1]:
%pip install -q huggingface_hub tokenizers scikit-learn tqdm rouge-score


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, math, random, gc, tempfile, zipfile, uuid, re, hashlib
from pathlib import Path
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download, HfFileSystem
import time
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
DATA_REPO = 'Exploration-Lab/CS-781-Capstone'
BUCKET_ROOT = 'hf://buckets/A-240031/legal-lm-phase1-bucket'
PHASE1_TAG = 'dual_attention'
MODEL_PATH = 'checkpoints/dual_attention_200M/model.pt'
TOKENIZER_PATH = 'tokenizer_dual_attention/tokenizer.json'
MODEL_CONFIG = {'d_model': 512, 'n_heads': 8, 'n_kv_heads': 4, 'n_layers': 12, 'd_ff': 2048, 'max_seq_len': 2048, 'n_experts': 8, 'top_k': 2, 'balance_alpha': 0.003}
MAX_SEQ_LEN = 2048
USE_GRADIENT_CHECKPOINTING = False
UNFREEZE_LAST_N_BLOCKS = 0  # pure LORMA, as in the supplied notebook
LORMA_LAST_N_BLOCKS = 2  # same attention targets as the supplied notebook
SOURCE_SPANS = 8  # evenly spaced non-overlapping source passages
VAL_MAX_DOCS = 100  # fixed held-out subset used for generation-based selection; None = full dev
LOG_EVERY_DOCS = 1000
USE_AMP = DEVICE == 'cuda'
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16

def autocast_context():
    return torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP)

DATA_PREFIX = 'Phase_7_PCR'
WARMUP_RATIO = 0.05
WARMUP_START_FACTOR = 0.1
LR_FACTOR = 0.5
LR_BAD_CHECKS = 2
LR_COOLDOWN_CHECKS = 1
LR_MIN = 1e-5
LR_LOSS_THRESHOLD = 0.001  # require 0.1% relative dev-loss improvement
assert 0 < MAX_SEQ_LEN <= MODEL_CONFIG['max_seq_len']
assert SOURCE_SPANS > 0 and LORMA_LAST_N_BLOCKS <= MODEL_CONFIG['n_layers']
assert UNFREEZE_LAST_N_BLOCKS == 0  # pure LoRMA
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
LR = 2e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
NUM_EPOCHS = 5  # retained from the reference; lower this to cap training time
PATIENCE = 10
MIN_DELTA = 0.001
ADAPTER_CONFIG = dict(r=16, alpha=32, dropout=0.05, do_pi=True)
UPLOAD_CHECKPOINTS = True
TECHNIQUE = 'lorma'
TRAIN_FRACTION = 1.0  # use all training documents after holding out validation

# All loading/saving locations are controlled here.
# Paths below are relative to BUCKET_ROOT; an absolute hf:// path also works.
SAVE_TAG = None  # None generates a descriptive tag; or enter your own run tag.
SUBMISSION_ZIP_NAME = None  # None uses SAVE_TAG + '.zip'; or enter a filename.
CHECKPOINT_PREFIX = 'checkpoints'
RESUME = False  # True resumes SAVE_TAG; set SAVE_TAG to the existing latest tag.
WORK_ROOT = Path('/kaggle/working/pcr_runs') if Path('/kaggle/working').exists() else Path.cwd() / 'pcr_runs'

if RESUME and SAVE_TAG is None:
    raise ValueError('To resume, set SAVE_TAG to the existing run tag.')
if SAVE_TAG is None:
    SAVE_TAG = f'pcr_rebuilt_{PHASE1_TAG}_{TECHNIQUE}_len{MAX_SEQ_LEN}'
if not isinstance(SAVE_TAG, str) or not re.fullmatch(r'[A-Za-z0-9_.-]+', SAVE_TAG) or SAVE_TAG in {'.', '..'}:
    raise ValueError('SAVE_TAG must be a single name containing letters, numbers, _, - or .')
BEST_TAG = SAVE_TAG + '_best'
if SUBMISSION_ZIP_NAME is None:
    SUBMISSION_ZIP_NAME = SAVE_TAG + '.zip'
if Path(SUBMISSION_ZIP_NAME).name != SUBMISSION_ZIP_NAME or not SUBMISSION_ZIP_NAME.endswith('.zip'):
    raise ValueError('SUBMISSION_ZIP_NAME must be a filename ending in .zip.')
WORK_DIR = WORK_ROOT / SAVE_TAG
WORK_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF')
    except Exception:
        pass
fs = HfFileSystem(token=HF_TOKEN)
if any(str(p).startswith('SET_') for p in (PHASE1_TAG, MODEL_PATH, TOKENIZER_PATH)):
    raise ValueError('Set the exact dual-attention Phase 1 tag, model checkpoint path and matching tokenizer path. The supplied reference notebook does not specify them.')

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(SEED)
print('Device:', DEVICE)
def bucket_path(path):
    if path.startswith('hf://'):
        return path
    return BUCKET_ROOT.rstrip('/') + '/' + path.lstrip('/')

def checkpoint_remote_path(tag):
    return bucket_path(f"{CHECKPOINT_PREFIX.strip('/')}/{tag}/checkpoint.pt")

def download_bucket_file(remote_path):
    # Identity-specific cache directory; refresh the bytes on each call.
    remote = bucket_path(remote_path)
    cache_key = hashlib.sha256(remote.encode()).hexdigest()[:20]
    local = WORK_ROOT / '_inputs' / cache_key / remote.rsplit('/', 1)[-1]
    local.parent.mkdir(parents=True, exist_ok=True)
    temporary = local.with_name(local.name + '.download')
    fs.get_file(remote, str(temporary))
    os.replace(temporary, local)
    return local

def assert_run_destination():
    if RESUME:
        return
    for tag in [SAVE_TAG, BEST_TAG]:
        local = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
        if local.exists() or (UPLOAD_CHECKPOINTS and fs.exists(checkpoint_remote_path(tag))):
            raise FileExistsError(
                f'Checkpoint already exists for {tag}. Choose a new SAVE_TAG, '
                'or set RESUME=True with the original latest SAVE_TAG.'
            )
    manifest = experiment_metadata('PCR')
    manifest.update(save_tag=SAVE_TAG, best_tag=BEST_TAG, submission_zip=SUBMISSION_ZIP_NAME)
    with open(WORK_DIR / 'run_config.json', 'w') as f:
        json.dump(manifest, f, indent=2)

print('Phase-1 model:', bucket_path(MODEL_PATH))
print('Tokenizer:', bucket_path(TOKENIZER_PATH))
print('Latest checkpoint:', checkpoint_remote_path(SAVE_TAG))
print('Best checkpoint:', checkpoint_remote_path(BEST_TAG))
print('Submission ZIP:', WORK_DIR / SUBMISSION_ZIP_NAME)

NEGATIVES_PER_QUERY = 2
TEMPERATURE = 0.07
ENCODE_BATCH_SIZE = 1
IS_ENCODER_DECODER = False
IS_MOE = True

Device: cuda
Phase-1 model: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/dual_attention_200M/model.pt
Tokenizer: hf://buckets/A-240031/legal-lm-phase1-bucket/tokenizer_dual_attention/tokenizer.json
Latest checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/pcr_rebuilt_dual_attention_lorma_len2048/checkpoint.pt
Best checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/pcr_rebuilt_dual_attention_lorma_len2048_best/checkpoint.pt
Submission ZIP: /kaggle/working/pcr_runs/pcr_rebuilt_dual_attention_lorma_len2048/pcr_rebuilt_dual_attention_lorma_len2048.zip


In [3]:
local_tokenizer = download_bucket_file(TOKENIZER_PATH)
tokenizer = Tokenizer.from_file(str(local_tokenizer))
tokenizer.no_padding()
tokenizer.no_truncation()
VOCAB_SIZE = tokenizer.get_vocab_size()
EMPTY_ID = tokenizer.token_to_id('<|eos|>')
PAD_ID = tokenizer.token_to_id('<|pad|>')
if EMPTY_ID is None:
    raise ValueError('Expected the Phase-1 <|eos|> token.')
if PAD_ID is None: PAD_ID = EMPTY_ID


EOS_ID = EMPTY_ID
BOS_ID = tokenizer.token_to_id('<|bos|>')
DECODER_START_ID = EOS_ID if BOS_ID is None else BOS_ID
PASSAGE_GAP_IDS = tokenizer.encode('\n[...]\n',add_special_tokens=False).ids
SOURCE_BUDGET = MAX_SEQ_LEN
print('Vocabulary:',VOCAB_SIZE,'| case token budget:',SOURCE_BUDGET)

(…)/tokenizer_dual_attention/tokenizer.json:   0%|          | 0.00/2.54M [00:00<?, ?B/s]

Vocabulary: 33887 | case token budget: 2048


In [4]:
def download_json(filename):
    path = hf_hub_download(repo_id=DATA_REPO, filename=filename, repo_type='dataset', token=HF_TOKEN)
    with open(path, encoding='utf-8') as f: return json.load(f)

def case_text(value):
    if isinstance(value,str): return value
    if isinstance(value,list): return '\n'.join(case_text(x) for x in value)
    if isinstance(value,dict):
        for name in ('text','sentence','document'):
            if name in value: return case_text(value[name])
    raise ValueError('Expected case text as a string or a list of sentences')

def normalize_cases(raw,labelled=False):
    if isinstance(raw,list): entries=[(None,r) for r in raw]
    elif isinstance(raw,dict) and 'id' in raw: entries=[(None,raw)]
    elif isinstance(raw,dict): entries=list(raw.items())
    else: raise ValueError('Expected a list or ID-to-record mapping')
    records=[]; seen=set()
    for key,value in entries:
        if isinstance(value,(str,list)) and key is not None: value={'text':value}
        if not isinstance(value,dict): raise ValueError('Each case record must be an object')
        ident=value.get('id',value.get('case_id',value.get('query_id',key)))
        if not isinstance(ident,str): raise ValueError('Expected string case IDs; preserve leading zeros')
        if key is not None and ident!=key: raise ValueError('Case ID does not match mapping key')
        if ident in seen: raise ValueError(f'Duplicate case ID: {ident}')
        seen.add(ident)
        text=case_text(value.get('text',value.get('document'))).strip()
        if not text: raise ValueError(f'{ident}: empty case text')
        record={'id':ident,'text':text}
        if labelled:
            if 'relevant_candidates' not in value:
                raise ValueError(f'{ident}: missing relevant_candidates field; available fields: {list(value)}')
            gold=value['relevant_candidates']
            if not isinstance(gold,list) or any(not isinstance(c,str) for c in gold):
                raise ValueError(f'{ident}: relevant_candidates must be a list of string IDs')
            record['relevant_candidates']=list(dict.fromkeys(gold))
        records.append(record)
    if not records: raise ValueError('No cases found')
    return records

data={}; candidate_pools={}; fingerprints={}
for split,folder in [('train','Train'),('dev','Dev'),('test','Test')]:
    raw_queries=download_json(f'{DATA_PREFIX}/{folder}/{split}.json')
    raw_candidates=download_json(f'{DATA_PREFIX}/{folder}/{split}_candidates.json')
    data[split]=normalize_cases(raw_queries,labelled=split!='test')
    candidate_pools[split]=normalize_cases(raw_candidates)
    pool_ids={r['id'] for r in candidate_pools[split]}
    if len(pool_ids)<7: raise ValueError(f'{split}: fewer than seven candidates')
    for row in data[split]:
        missing=set(row.get('relevant_candidates',[]))-pool_ids
        if missing: raise ValueError(f'{split}/{row["id"]}: relevant IDs absent from this split pool: {sorted(missing)[:5]}')
    digest=hashlib.sha256()
    for records in (data[split],candidate_pools[split]):
        for row in records: digest.update(json.dumps(row,sort_keys=True,ensure_ascii=False).encode())
    fingerprints[split]=digest.hexdigest()
    print(split,'queries:',len(data[split]),'candidates:',len(pool_ids),
          'empty relevance lists:',sum(not r.get('relevant_candidates',[]) for r in data[split]) if split!='test' else 'private')
DATA_FINGERPRINT=fingerprints
train_rows=[r for r in data['train'] if r['relevant_candidates']]
print('Training queries with positives:',len(train_rows),'| skipped empty relevance lists:',len(data['train'])-len(train_rows))
if not train_rows: raise ValueError('No training queries have relevant candidates')
if TRAIN_FRACTION<1:
    train_rows=random.Random(SEED).sample(train_rows,max(1,round(len(train_rows)*TRAIN_FRACTION)))
dev_rows=data['dev'] if VAL_MAX_DOCS is None else data['dev'][:VAL_MAX_DOCS]

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

train queries: 827 candidates: 4320 empty relevance lists: 10


dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

... (3 outputs left out)

test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

test queries: 237 candidates: 1727 empty relevance lists: private
Training queries with positives: 817 | skipped empty relevance lists: 10


In [5]:
def select_source_tokens(tokens,budget=SOURCE_BUDGET,spans=SOURCE_SPANS):
    if len(tokens)<=budget: return list(tokens)
    spans=min(spans,budget)
    content_budget=budget-(spans-1)*len(PASSAGE_GAP_IDS)
    if content_budget<spans: raise ValueError('Source budget too small for passage markers')
    widths=[content_budget//spans+int(i<content_budget%spans) for i in range(spans)]
    omitted=len(tokens)-content_budget
    if spans==1: return tokens[:budget]
    selected=[];consumed=0
    for i,width in enumerate(widths):
        start=consumed+i*omitted//(spans-1)
        if i:selected.extend(PASSAGE_GAP_IDS)
        selected.extend(tokens[start:start+width]);consumed+=width
    return selected

query_tokens={};candidate_tokens={}
for split in ('train','dev','test'):
    query_tokens[split]={r['id']:select_source_tokens(tokenizer.encode(r['text'],add_special_tokens=False).ids) or [EOS_ID]
                         for r in tqdm(data[split],desc=f'Tokenize {split} queries')}
    candidate_tokens[split]={r['id']:select_source_tokens(tokenizer.encode(r['text'],add_special_tokens=False).ids) or [EOS_ID]
                             for r in tqdm(candidate_pools[split],desc=f'Tokenize {split} candidates')}

class RetrievalDataset(Dataset):
    def __len__(self):return len(train_rows)
    def __getitem__(self,i):return train_rows[i]

def collate_retrieval(rows):return rows

def make_loader():
    return DataLoader(RetrievalDataset(),batch_size=BATCH_SIZE,shuffle=True,
                      collate_fn=collate_retrieval,num_workers=0)

train_candidate_ids=list(candidate_tokens['train'])
def sample_training_candidates(row):
    positive=random.choice(row['relevant_candidates']);gold=set(row['relevant_candidates'])
    if len(gold)>=len(train_candidate_ids):raise ValueError('Training query has no negative candidates')
    negatives=[]; used=set()
    for _ in range(NEGATIVES_PER_QUERY):
        # Rejection sampling avoids storing a candidate-pool copy for every query.
        for attempt in range(100):
            candidate=random.choice(train_candidate_ids)
            if candidate not in gold and candidate not in used:break
        else:
            eligible=[cid for cid in train_candidate_ids if cid not in gold and cid not in used]
            if not eligible:eligible=[cid for cid in train_candidate_ids if cid not in gold]
            candidate=random.choice(eligible)
        negatives.append(candidate);used.add(candidate)
    return [positive]+negatives

Tokenize train queries:   0%|          | 0/827 [00:00<?, ?it/s]

Tokenize train candidates:   0%|          | 0/4320 [00:00<?, ?it/s]

Tokenize dev queries:   0%|          | 0/118 [00:00<?, ?it/s]

Tokenize dev candidates:   0%|          | 0/1023 [00:00<?, ?it/s]

Tokenize test queries:   0%|          | 0/237 [00:00<?, ?it/s]

Tokenize test candidates:   0%|          | 0/1727 [00:00<?, ?it/s]

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# RoPE
# ============================================================

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len=2048, base=10000):
        super().__init__()

        inv_freq = 1.0 / (
            base ** (
                torch.arange(0, head_dim, 2).float() / head_dim
            )
        )

        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)

        self.register_buffer(
            "cos",
            freqs.cos(),
            persistent=False,
        )

        self.register_buffer(
            "sin",
            freqs.sin(),
            persistent=False,
        )

    def forward(self, q, k):

        T = q.size(2)

        cos = self.cos[:T].unsqueeze(0).unsqueeze(0)
        sin = self.sin[:T].unsqueeze(0).unsqueeze(0)

        q1, q2 = q[..., ::2], q[..., 1::2]
        k1, k2 = k[..., ::2], k[..., 1::2]

        q_rot = torch.stack(
            [
                q1 * cos - q2 * sin,
                q1 * sin + q2 * cos,
            ],
            dim=-1,
        ).flatten(-2)

        k_rot = torch.stack(
            [
                k1 * cos - k2 * sin,
                k1 * sin + k2 * cos,
            ],
            dim=-1,
        ).flatten(-2)

        return q_rot, k_rot


# ============================================================
# SwiGLU Expert
# ============================================================

class SwiGLU(nn.Module):

    def __init__(self, d_model, d_ff):
        super().__init__()

        self.gate_proj = nn.Linear(
            d_model,
            d_ff,
            bias=False,
        )

        self.up_proj = nn.Linear(
            d_model,
            d_ff,
            bias=False,
        )

        self.down_proj = nn.Linear(
            d_ff,
            d_model,
            bias=False,
        )

    def forward(self, x):

        return self.down_proj(
            F.silu(self.gate_proj(x))
            * self.up_proj(x)
        )


# ============================================================
# Mixture of Experts
# ============================================================
class MoE(nn.Module):

    def __init__(
        self,
        d_model,
        d_ff,
        n_experts=8,
        top_k=2,
        balance_alpha=0.003,
    ):
        super().__init__()

        self.n_experts = n_experts
        self.top_k = top_k
        self.balance_alpha = balance_alpha

        # Router chooses experts BEFORE any expert is evaluated
        self.router = nn.Linear(
            d_model,
            n_experts,
            bias=False,
        )

        expert_d_ff = d_ff // top_k

        self.experts = nn.ModuleList([
            SwiGLU(
                d_model,
                expert_d_ff,
            )
            for _ in range(n_experts)
        ])

    def forward(self, x, valid_mask=None):

        # x: [B, T, D]

        # ------------------------------------------------
        # Batch representation
        # ------------------------------------------------

        batch_repr = (x.mean(dim=(0, 1)) if valid_mask is None else
            (x * valid_mask[..., None]).sum(dim=(0, 1)) / valid_mask.sum().clamp_min(1))  # [D]

        # ------------------------------------------------
        # ROUTE FIRST
        # ------------------------------------------------

        router_logits = self.router(
            batch_repr
        )  # [E]

        router_probs = F.softmax(
            router_logits,
            dim=-1,
        )

        topk_probs, topk_indices = torch.topk(
            router_probs,
            self.top_k,
            dim=-1,
        )

        # Normalize selected weights
        topk_probs = topk_probs / (
            topk_probs.sum() + 1e-9
        )

        # ------------------------------------------------
        # ONLY NOW evaluate selected experts
        # ------------------------------------------------

        output = 0.0

        for i in range(self.top_k):

            expert_idx = topk_indices[i].item()
            weight = topk_probs[i]

            expert_output = self.experts[
                expert_idx
            ](x)

            output = (
                output
                + weight * expert_output
            )

        # ------------------------------------------------
        # Batch routing loss
        # ------------------------------------------------

        # Router probability distribution for this batch.
        # Used to discourage permanent collapse to
        # a small subset of experts.
        balance_loss = (
            self.balance_alpha
            * self.n_experts
            * torch.sum(
                router_probs
                * router_probs
            )
        )

        return output, balance_loss

# ============================================================
# GQA + SDPA
# ============================================================
class GQAAttention(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        n_kv_heads,
        local_window=64,
        global_stride=128,
        max_seq_len=2048,
    ):
        super().__init__()

        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.local_window = local_window
        self.global_stride = global_stride
        self.max_seq_len = max_seq_len

        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(
            d_model,
            n_heads * self.head_dim,
            bias=False,
        )

        self.k_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.v_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=False,
        )

        self.rope = RotaryEmbedding(
            self.head_dim,
            max_seq_len=max_seq_len,
        )

    def _build_sparse_mask(self, T, device):
        """
        Allowed attention for query position i:

        1. Previous local_window positions
        2. All previous global positions
           (every global_stride tokens)
        3. Itself

        Future positions are never allowed.
        """

        positions = torch.arange(
            T,
            device=device,
        )

        # i = query position
        # j = key position
        i = positions[:, None]
        j = positions[None, :]

        # Causal condition
        causal = j <= i

        # Local attention
        local = j >= (i - self.local_window + 1)

        # Global landmark positions
        global_pos = (j % self.global_stride) == 0

        # Allowed if local OR global, while remaining causal
        allowed = causal & (local | global_pos)

        # SDPA bool mask:
        # True  = allowed to attend
        # False = masked
        return allowed

    def forward(self, x, valid_mask=None):

        B, T, _ = x.shape

        # --------------------------------------------------
        # Q, K, V projections
        # --------------------------------------------------

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # --------------------------------------------------
        # Reshape into heads
        # --------------------------------------------------

        q = q.view(
            B,
            T,
            self.n_heads,
            self.head_dim,
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.n_kv_heads,
            self.head_dim,
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.n_kv_heads,
            self.head_dim,
        ).transpose(1, 2)

        # --------------------------------------------------
        # RoPE
        # --------------------------------------------------

        q, k = self.rope(q, k)

        # --------------------------------------------------
        # GQA: expand K/V heads
        # --------------------------------------------------

        repeat_factor = (
            self.n_heads // self.n_kv_heads
        )

        k = k.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        v = v.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        # --------------------------------------------------
        # Local + Global sparse attention
        # --------------------------------------------------

        attn_mask = self._build_sparse_mask(
            T,
            x.device,
        )

        if valid_mask is not None:
            attn_mask = attn_mask[None, None] & valid_mask[:, None, None, :]

        attn_out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            is_causal=False,
        )

        # --------------------------------------------------
        # Merge heads
        # --------------------------------------------------

        attn_out = (
            attn_out
            .transpose(1, 2)
            .contiguous()
            .view(B, T, -1)
        )

        return self.out_proj(attn_out)


# ============================================================
# Decoder Block with MoE
# ============================================================

class DecoderBlock(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        n_kv_heads,
        d_ff,
        n_experts=8,
        top_k=2,
        balance_alpha=0.003,
    ):
        super().__init__()

        self.norm1 = nn.RMSNorm(
            d_model
        )

        self.attention = GQAAttention(
            d_model,
            n_heads,
            n_kv_heads,
        )

        self.norm2 = nn.RMSNorm(
            d_model
        )

        self.moe = CausalMoE(
            d_model=d_model,
            d_ff=d_ff,
            n_experts=n_experts,
            top_k=top_k,
            balance_alpha=balance_alpha,
        )

    def forward(self, x, valid_mask=None):

        # Attention
        x = x + self.attention(
            self.norm1(x), valid_mask=valid_mask
        )

        # MoE FFN
        moe_output, balance_loss = self.moe(
            self.norm2(x), valid_mask=valid_mask
        )

        x = x + moe_output

        return x, balance_loss


# ============================================================
# Decoder-only MoE Language Model
# ============================================================

class EfficientDecoderLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=768,
        n_heads=12,
        n_kv_heads=4,
        n_layers=12,
        d_ff=3072,
        max_seq_len=2048,

        # MoE
        n_experts=8,
        top_k=2,
        balance_alpha=0.003,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model

        self.n_experts = n_experts
        self.top_k = top_k
        self.balance_alpha = balance_alpha

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model,
        )

        self.blocks = nn.ModuleList([
            DecoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                d_ff=d_ff,
                n_experts=n_experts,
                top_k=top_k,
                balance_alpha=balance_alpha,
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.RMSNorm(
            d_model
        )

        # Tied LM head
        self.lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False,
        )

        self.lm_head.weight = (
            self.token_embedding.weight
        )

        self.apply(
            self._init_weights
        )

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):

            nn.init.xavier_uniform_(
                module.weight
            )

        elif isinstance(module, nn.Embedding):

            nn.init.xavier_uniform_(
                module.weight
            )

        elif isinstance(module, nn.RMSNorm):

            nn.init.ones_(
                module.weight
            )

    def forward(
        self,
        input_ids,
        return_aux_loss=False,
    ):

        x = self.token_embedding(
            input_ids
        )

        total_balance_loss = x.new_zeros(
            ()
        )

        for block in self.blocks:

            x, balance_loss = block(x)

            total_balance_loss = (
                total_balance_loss
                + balance_loss
            )

        x = self.final_norm(x)

        logits = self.lm_head(x)

        if return_aux_loss:

            return (
                logits,
                total_balance_loss,
            )

        return logits


In [7]:
class LoRMALinear(nn.Module):
    def __init__(self, original_linear, r=16, alpha=32, dropout=0.05, do_pi=True):
        super().__init__()
        self.original_linear = original_linear
        self.in_features = original_linear.in_features
        self.out_features = original_linear.out_features
        self.scaling, self.do_pi = alpha / r, do_pi
        self.dropout = nn.Dropout(dropout)
        for p in original_linear.parameters(): p.requires_grad = False
        W = original_linear.weight
        self.A = nn.Parameter(W.new_empty(r, self.out_features))
        self.B = nn.Parameter(W.new_zeros(self.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def effective_weight(self):
        W = self.original_linear.weight
        delta = self.B @ self.A
        if self.do_pi:
            rows = torch.arange(self.out_features, device=W.device)[:, None]
            cols = torch.arange(self.out_features, device=W.device)[None, :]
            delta = delta.gather(1, (cols - rows) % self.out_features)
        return W + self.scaling * (delta @ W)
    def forward(self, x):
        W = self.original_linear.weight
        W_eff = self.effective_weight()
        return F.linear(x, W, self.original_linear.bias) + F.linear(self.dropout(x), W_eff - W)


from torch.utils.checkpoint import checkpoint as activation_checkpoint
from contextlib import contextmanager

class CausalMoE(MoE):
    def forward(self, x, valid_mask=None):
        # Preserve the pretrained router and expert tensors, but route each token
        # independently so future target tokens and other batch rows cannot leak.
        B, T, D = x.shape
        flat = x.reshape(-1, D)
        valid = (torch.ones(B*T, dtype=torch.bool, device=x.device) if valid_mask is None
                 else valid_mask.reshape(-1).bool())
        probs = F.softmax(self.router(flat).float(), dim=-1)
        weights, indices = torch.topk(probs, self.top_k, dim=-1)
        weights = weights / weights.sum(dim=-1, keepdim=True).clamp_min(1e-9)
        output = torch.zeros_like(flat)
        # Dispatch only selected valid tokens to each expert.
        for expert_index, expert in enumerate(self.experts):
            token_ids, slots = ((indices == expert_index) & valid[:,None]).nonzero(as_tuple=True)
            if token_ids.numel() == 0: continue
            expert_output = expert(flat[token_ids])
            weighted = expert_output * weights[token_ids,slots,None].to(expert_output.dtype)
            output = output.index_add(0, token_ids, weighted.to(output.dtype))
        mean_probs = (probs * valid[:,None]).sum(dim=0) / valid.sum().clamp_min(1)
        balance_loss = self.balance_alpha * self.n_experts * mean_probs.square().sum()
        return output.view(B,T,D), balance_loss

class RetrievalBackbone(EfficientDecoderLM):
    def forward(self, input_ids, valid_mask=None, return_aux_loss=False):
        x = self.token_embedding(input_ids)
        balance_sum = x.new_zeros(())
        for block in self.blocks:
            if self.training and USE_GRADIENT_CHECKPOINTING:
                x, balance = activation_checkpoint(block, x, valid_mask=valid_mask,
                    use_reentrant=False, preserve_rng_state=True)
            else: x, balance = block(x, valid_mask=valid_mask)
            balance_sum = balance_sum + balance
        logits = self.lm_head(self.final_norm(x))
        return (logits, balance_sum) if return_aux_loss else logits

    def forward_cached(self, input_ids, past=None):
        # Inference only. Keep RoPE positions relative to the complete prompt.
        x = self.token_embedding(input_ids)
        offset = 0 if past is None else past[0][0].size(2)
        T = input_ids.size(1)
        if offset + T > MAX_SEQ_LEN: raise ValueError('Generation exceeds Phase 1 context')
        new_past = []
        for i, block in enumerate(self.blocks):
            a = block.attention
            h = block.norm1(x)
            B = h.size(0)
            q = a.q_proj(h).view(B,T,a.n_heads,a.head_dim).transpose(1,2)
            k = a.k_proj(h).view(B,T,a.n_kv_heads,a.head_dim).transpose(1,2)
            v = a.v_proj(h).view(B,T,a.n_kv_heads,a.head_dim).transpose(1,2)
            cos = a.rope.cos[offset:offset+T][None,None]
            sin = a.rope.sin[offset:offset+T][None,None]
            def rotate(z):
                z1,z2 = z[...,::2],z[...,1::2]
                return torch.stack((z1*cos-z2*sin,z1*sin+z2*cos),dim=-1).flatten(-2)
            q,k = rotate(q),rotate(k)
            if past is not None:
                k = torch.cat((past[i][0],k),dim=2)
                v = torch.cat((past[i][1],v),dim=2)
            new_past.append((k,v))
            repeat = a.n_heads // a.n_kv_heads
            query_positions = (offset + torch.arange(T, device=x.device))[:,None]
            key_positions = torch.arange(k.size(2), device=x.device)[None,:]
            causal = key_positions <= query_positions
            local = key_positions >= query_positions - a.local_window + 1
            global_landmarks = (key_positions % a.global_stride) == 0
            mask = causal & (local | global_landmarks)
            out = F.scaled_dot_product_attention(q,k.repeat_interleave(repeat,dim=1),v.repeat_interleave(repeat,dim=1),attn_mask=mask[None,None])
            x = x + a.out_proj(out.transpose(1,2).contiguous().view(B,T,-1))
            moe_output, _ = block.moe(block.norm2(x))
            x = x + moe_output
        return self.lm_head(self.final_norm(x[:,-1:])), new_past

phase1_path = download_bucket_file(MODEL_PATH)

def build_retrieval_model():
    model = RetrievalBackbone(VOCAB_SIZE, **MODEL_CONFIG)
    checkpoint = torch.load(phase1_path, map_location='cpu', weights_only=False)
    if 'model_state_dict' in checkpoint: state = checkpoint['model_state_dict']
    elif 'model' in checkpoint: state = checkpoint['model']
    elif 'state_dict' in checkpoint: state = checkpoint['state_dict']
    elif all(isinstance(v, torch.Tensor) for v in checkpoint.values()): state = checkpoint
    else: raise ValueError(f'Cannot identify weights: {list(checkpoint)}')
    # Restore the original vocabulary head too; no Phase 5 classification weights.
    state = dict(state)
    if 'lm_head.weight' not in state and 'token_embedding.weight' in state:
        state['lm_head.weight'] = state['token_embedding.weight']
    if not torch.equal(state['lm_head.weight'], state['token_embedding.weight']):
        raise ValueError('Checkpoint LM head and embedding disagree with the tied Phase 1 architecture')
    model.load_state_dict(state, strict=True)
    del state, checkpoint
    for p in model.parameters(): p.requires_grad = False
    for block in model.blocks[-LORMA_LAST_N_BLOCKS:]:
        for name in ['q_proj', 'k_proj', 'v_proj', 'out_proj']:
            setattr(block.attention, name, LoRMALinear(getattr(block.attention, name), **ADAPTER_CONFIG))
    model = model.to(DEVICE)
    assert model.lm_head.weight is model.token_embedding.weight
    print('Total/trainable parameters:', sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad))
    return model



def retrieval_hidden(model,ids,mask):
    if IS_ENCODER_DECODER:
        memory=model.encode(ids,encoder_mask=mask)
        mean=(memory.float()*mask[...,None]).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)
        # A one-token decoder readout uses the pretrained decoder and cross-attention too.
        x=model.token_embedding(torch.full((len(ids),1),DECODER_START_ID,device=ids.device,dtype=torch.long))
        for block in model.decoder_blocks:
            if model.training and USE_GRADIENT_CHECKPOINTING:
                x,_=activation_checkpoint(block,x,memory,encoder_mask=mask,use_reentrant=False,preserve_rng_state=True)
            else:x,_=block(x,memory,encoder_mask=mask)
        return (mean+model.decoder_norm(x[:,0]).float())/2
    x=model.token_embedding(ids)
    for block in model.blocks:
        if IS_MOE:
            if model.training and USE_GRADIENT_CHECKPOINTING:
                x,_=activation_checkpoint(block,x,valid_mask=mask,use_reentrant=False,preserve_rng_state=True)
            else:x,_=block(x,valid_mask=mask)
        else:
            if model.training and USE_GRADIENT_CHECKPOINTING:
                x=activation_checkpoint(block,x,use_reentrant=False,preserve_rng_state=True)
            else:x=block(x)
    x=model.final_norm(x)
    return (x.float()*mask[...,None]).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)

def embed_sequences(model,sequences):
    vectors=[]
    for start in range(0,len(sequences),ENCODE_BATCH_SIZE):
        part=sequences[start:start+ENCODE_BATCH_SIZE];width=max(map(len,part))
        ids=torch.full((len(part),width),PAD_ID,device=DEVICE,dtype=torch.long)
        mask=torch.zeros_like(ids,dtype=torch.bool)
        for i,seq in enumerate(part):
            ids[i,:len(seq)]=torch.tensor(seq,device=DEVICE);mask[i,:len(seq)]=True
        with autocast_context():v=retrieval_hidden(model,ids,mask)
        vectors.append(F.normalize(v.float(),dim=-1))
    return torch.cat(vectors)

def retrieval_loss_sum(model,rows):
    total=None
    for row in rows:
        candidates=sample_training_candidates(row)
        sequences=[query_tokens['train'][row['id']]]+[candidate_tokens['train'][cid] for cid in candidates]
        embeddings=embed_sequences(model,sequences)
        scores=(embeddings[:1]@embeddings[1:].T)/TEMPERATURE
        loss=F.cross_entropy(scores,torch.zeros(1,dtype=torch.long,device=DEVICE))
        total=loss if total is None else total+loss
    return total,len(rows)

@torch.no_grad()
def rank_cases(model,split,rows=None):
    was_training=model.training;model.eval()
    try:
        if rows is None:rows=data[split]
        ids=list(candidate_tokens[split]);embeddings=[];started=time.perf_counter()
        for start in tqdm(range(0,len(ids),ENCODE_BATCH_SIZE),desc=f'Encode {split} pool'):
            batch=ids[start:start+ENCODE_BATCH_SIZE]
            embeddings.append(embed_sequences(model,[candidate_tokens[split][cid] for cid in batch]).cpu())
        matrix=torch.cat(embeddings);predictions={};hits=gold_count=0
        for row in tqdm(rows,desc=f'Rank {split}'):
            q=embed_sequences(model,[query_tokens[split][row['id']]])[0].cpu()
            # Sorting stable ties follows the candidate file order.
            selected=torch.argsort(matrix@q,descending=True,stable=True)[:7].tolist()
            predictions[row['id']]=[ids[i] for i in selected]
            if split!='test':
                gold=set(row['relevant_candidates']);gold_count+=len(gold)
                hits+=len(gold.intersection(predictions[row['id']]))
        p=hits/(7*len(rows)) if rows else 0.0;r=hits/gold_count if gold_count else 0.0
        metrics=dict(micro_precision_at_7=p,micro_recall_at_7=r,micro_f1_at_7=2*p*r/(p+r) if p+r else 0.0,
                     hits=hits,gold_relevant=gold_count,queries=len(rows),ranking_seconds=time.perf_counter()-started)
        return predictions,metrics
    finally:model.train(was_training)

(…)checkpoints/dual_attention_200M/model.pt:   0%|          | 0.00/712M [00:00<?, ?B/s]

In [8]:
def experiment_metadata(domain='PCR'):
    # Epoch target is deliberately not part of compatibility: it may be extended on resume.
    return dict(domain=domain,phase1_tag=PHASE1_TAG,model_path=MODEL_PATH,tokenizer_path=TOKENIZER_PATH,
                model_config=MODEL_CONFIG,adapter_config=ADAPTER_CONFIG,lorma_last_n_blocks=LORMA_LAST_N_BLOCKS,
                source_budget=SOURCE_BUDGET,source_spans=SOURCE_SPANS,data_fingerprint=DATA_FINGERPRINT,
                batch_size=BATCH_SIZE,grad_accum_steps=GRAD_ACCUM_STEPS,lr=LR,weight_decay=WEIGHT_DECAY,
                negatives_per_query=NEGATIVES_PER_QUERY,temperature=TEMPERATURE,seed=SEED,
                train_fraction=TRAIN_FRACTION,val_max_docs=VAL_MAX_DOCS,
                scheduler=dict(warmup_ratio=WARMUP_RATIO,start_factor=WARMUP_START_FACTOR,factor=LR_FACTOR,
                               bad_checks=LR_BAD_CHECKS,cooldown=LR_COOLDOWN_CHECKS,min_lr=LR_MIN,threshold=LR_LOSS_THRESHOLD))

def save_retrieval_checkpoint(tag, model, optimizer, epoch, extra, domain):
    folder = WORK_DIR / 'checkpoints' / tag
    folder.mkdir(parents=True, exist_ok=True)
    rng = {'python': random.getstate(), 'torch': torch.get_rng_state()}
    if torch.cuda.is_available(): rng['cuda'] = torch.cuda.get_rng_state_all()
    payload = dict(model=model.state_dict(), optimizer=optimizer.state_dict(), rng=rng,
                   epoch=epoch, extra=extra, metadata=experiment_metadata(domain))
    tmp = folder / 'checkpoint.tmp'
    torch.save(payload, tmp)
    os.replace(tmp, folder / 'checkpoint.pt')
    if UPLOAD_CHECKPOINTS:
        fs.put_file(str(folder / 'checkpoint.pt'), checkpoint_remote_path(tag))

def load_retrieval_checkpoint(tag, model, domain, optimizer=None, restore_rng=False):
    path = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
    if not path.exists():
        if UPLOAD_CHECKPOINTS:
            path = download_bucket_file(checkpoint_remote_path(tag))
        elif not path.exists():
            raise FileNotFoundError(path)
    payload = torch.load(path, map_location='cpu', weights_only=False)  # own trusted checkpoints only
    if payload['metadata'] != experiment_metadata(domain):
        changed = [k for k in set(payload['metadata']) | set(experiment_metadata(domain)) if payload['metadata'].get(k) != experiment_metadata(domain).get(k)]
        raise ValueError('Checkpoint configuration mismatch: ' + ', '.join(sorted(changed)))
    model.load_state_dict(payload['model'])
    if optimizer is not None: optimizer.load_state_dict(payload['optimizer'])
    if restore_rng:
        random.setstate(payload['rng']['python'])
        torch.set_rng_state(payload['rng']['torch'])
        if torch.cuda.is_available() and 'cuda' in payload['rng']:
            torch.cuda.set_rng_state_all(payload['rng']['cuda'])
    return payload



class ValidationLRScheduler:
    def __init__(self, optimizer, total_updates):
        assert total_updates > 0 and 0 < WARMUP_RATIO < 1
        assert 0 < WARMUP_START_FACTOR <= 1 and 0 < LR_FACTOR < 1
        assert LR_BAD_CHECKS >= 1 and 0 <= LR_MIN <= LR
        self.optimizer = optimizer
        self.updates = 0
        self.warmup_updates = max(1, math.ceil(total_updates * WARMUP_RATIO))
        self.peak_lrs = [g['lr'] for g in optimizer.param_groups]
        # PyTorch reduces after num_bad_epochs > patience: 1 means two bad checks.
        self.plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=LR_FACTOR, patience=LR_BAD_CHECKS-1,
            threshold=LR_LOSS_THRESHOLD, threshold_mode='rel',
            cooldown=LR_COOLDOWN_CHECKS, min_lr=LR_MIN)
        self._set_warmup_lr()

    def _set_warmup_lr(self):
        factor = WARMUP_START_FACTOR + (1-WARMUP_START_FACTOR) * self.updates/self.warmup_updates
        for group, peak in zip(self.optimizer.param_groups, self.peak_lrs):
            group['lr'] = peak * factor

    def optimizer_step_completed(self):
        self.updates += 1
        if self.updates <= self.warmup_updates: self._set_warmup_lr()

    def validation_step(self, loss):
        if not math.isfinite(loss): raise ValueError('Non-finite validation loss')
        # Warmup and plateau never fight over the learning rate.
        if self.updates >= self.warmup_updates: self.plateau.step(loss)

    def state_dict(self):
        return dict(updates=self.updates, warmup_updates=self.warmup_updates,
                    peak_lrs=self.peak_lrs, plateau=self.plateau.state_dict(),
                    current_lrs=[g['lr'] for g in self.optimizer.param_groups])

    def load_state_dict(self, state):
        self.warmup_updates = state['warmup_updates']  # restore the original horizon when extending epochs
        self.updates = state['updates']
        self.peak_lrs = state['peak_lrs']
        self.plateau.load_state_dict(state['plateau'])
        for group, lr in zip(self.optimizer.param_groups, state['current_lrs']): group['lr'] = lr

In [9]:
def train_retrieval(domain='PCR'):
    assert_run_destination();seed_everything(SEED)
    model=build_retrieval_model()
    params=[p for p in model.parameters() if p.requires_grad]
    optimizer=torch.optim.AdamW(params,lr=LR,weight_decay=WEIGHT_DECAY)
    scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP and AMP_DTYPE==torch.float16)
    loader=make_loader();updates_per_epoch=math.ceil(len(loader)/GRAD_ACCUM_STEPS)
    scheduler=ValidationLRScheduler(optimizer,NUM_EPOCHS*updates_per_epoch)
    start=0;extra=dict(best_f1=-1.0,patience_reference=-1.0,stale=0,history=[],best_tag=BEST_TAG)
    if RESUME:
        payload=load_retrieval_checkpoint(SAVE_TAG,model,domain,optimizer,restore_rng=True)
        start=payload['epoch'];extra=payload['extra']
        scaler.load_state_dict(extra['scaler']);scheduler.load_state_dict(extra['scheduler'])
        del payload
        print('Resume at completed epoch:',start,'| target epochs:',NUM_EPOCHS)
    for epoch in range(start,NUM_EPOCHS):
        if extra['stale']>=PATIENCE:break
        model.train();optimizer.zero_grad(set_to_none=True)
        train_sum=train_n=group_n=seen=0
        for step,rows in enumerate(tqdm(loader,desc=f'PCR epoch {epoch+1}')):
            loss_sum,count=retrieval_loss_sum(model,rows)
            if not torch.isfinite(loss_sum):raise FloatingPointError('Nonfinite retrieval loss')
            scaler.scale(loss_sum).backward()
            train_sum+=loss_sum.item();train_n+=count;group_n+=count;seen+=count
            if (step+1)%GRAD_ACCUM_STEPS==0 or step+1==len(loader):
                scaler.unscale_(optimizer)
                for p in params:
                    if p.grad is not None:p.grad.div_(group_n)
                torch.nn.utils.clip_grad_norm_(params,GRAD_CLIP)
                previous=scaler.get_scale();scaler.step(optimizer);scaler.update()
                if scaler.get_scale()>=previous:scheduler.optimizer_step_completed()
                optimizer.zero_grad(set_to_none=True);group_n=0
            if seen and seen%LOG_EVERY_DOCS<len(rows):print('Queries:',seen,'| loss:',train_sum/train_n)
        predictions,metrics=rank_cases(model,'dev',dev_rows)
        f1=metrics['micro_f1_at_7'];scheduler.validation_step(1.0-f1)
        improved=f1>extra['best_f1']
        if improved:extra['best_f1']=f1;extra['best_metrics']=metrics
        if f1>extra['patience_reference']+MIN_DELTA:extra['patience_reference']=f1;extra['stale']=0
        else:extra['stale']+=1
        extra['history'].append(dict(epoch=epoch+1,train_loss=train_sum/train_n,lr=optimizer.param_groups[0]['lr'],**metrics))
        extra['scaler']=scaler.state_dict();extra['scheduler']=scheduler.state_dict()
        print('Dev retrieval:',metrics)
        if improved:save_retrieval_checkpoint(BEST_TAG,model,optimizer,epoch+1,extra,domain)
        save_retrieval_checkpoint(SAVE_TAG,model,optimizer,epoch+1,extra,domain)
        with open(WORK_DIR/'PCR_history.json','w') as f:json.dump(extra['history'],f,indent=2)
    if not extra['history']:raise ValueError('No trained checkpoint available; NUM_EPOCHS must be positive')
    load_retrieval_checkpoint(BEST_TAG,model,domain)
    with open(WORK_DIR/'PCR_best_dev_metrics.json','w') as f:json.dump(extra['best_metrics'],f,indent=2)
    # Retain latest for exact continuation; it differs from the best validation epoch.
    del optimizer
    return model,extra['history']

In [10]:
def validate_predictions(predictions,documents,pool):
    if set(predictions)!={row['id'] for row in documents}:raise ValueError('Prediction query IDs do not match Test queries')
    allowed={row['id'] for row in pool}
    for qid,ids in predictions.items():
        if len(ids)!=7 or len(set(ids))!=7:raise ValueError(f'{qid}: expected seven unique IDs')
        if not set(ids)<=allowed:raise ValueError(f'{qid}: prediction outside the Test pool')

model,history=train_retrieval()
predictions,_=rank_cases(model,'test')
validate_predictions(predictions,data['test'],candidate_pools['test'])
with open(WORK_DIR/'prediction.json','w',encoding='utf-8') as f:json.dump(predictions,f,ensure_ascii=False,indent=2)

Total/trainable parameters: 177942528 98304


PCR epoch 1:   0%|          | 0/409 [00:00<?, ?it/s]

Encode dev pool:   0%|          | 0/1023 [00:00<?, ?it/s]

Rank dev:   0%|          | 0/100 [00:00<?, ?it/s]

Dev retrieval: {'micro_precision_at_7': 0.017142857142857144, 'micro_recall_at_7': 0.015483870967741935, 'micro_f1_at_7': 0.016271186440677966, 'hits': 12, 'gold_relevant': 775, 'queries': 100, 'ranking_seconds': 241.63929978099986}


Processing Files (0 / 0): |          |  0.00B /  0.00B            

... (34 outputs left out)

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

Encode test pool:   0%|          | 0/1727 [00:00<?, ?it/s]

Rank test:   0%|          | 0/237 [00:00<?, ?it/s]

In [11]:
# Packaging can be rerun without retraining.
json_path=WORK_DIR/'prediction.json'
with open(json_path,encoding='utf-8') as f:predictions=json.load(f)
validate_predictions(predictions,data['test'],candidate_pools['test'])
zip_path=WORK_DIR/SUBMISSION_ZIP_NAME
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:z.write(json_path,arcname='prediction.json')
print('Submission:',json_path,'| ZIP:',zip_path)

Submission: /kaggle/working/pcr_runs/pcr_rebuilt_dual_attention_lorma_len2048/prediction.json | ZIP: /kaggle/working/pcr_runs/pcr_rebuilt_dual_attention_lorma_len2048/pcr_rebuilt_dual_attention_lorma_len2048.zip
